# 🌳 AI-Based Illegal Deforestation Detection System
### Deep Learning Pipeline · Segmentation · Change Detection · Temporal Modeling · Risk Prediction

---
**Framework:** PyTorch + segmentation_models_pytorch  
**Data:** Kaggle dataset with robust fallback to synthetic Sentinel-like imagery  
**Fixes Applied:**
- ✅ Fixed `DeforestationDataset._build_multichannel` indentation (was completely broken)
- ✅ Added missing `__getitem__` method to `DeforestationDataset`
- ✅ Added `rasterio` to install list
- ✅ Fixed empty `records=[]` in dataset test → uses actual records
- ✅ Removed duplicate model/dataset definitions across sections
- ✅ Fixed `compress_model` tuple return handling
- ✅ Enhanced visualisations throughout (dashboards, heatmaps, confusion matrices, NDVI overlays)
---

## 📦 Section 1 — Installs & Environment Setup

In [ ]:
# ── Install required packages ────────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('kagglehub[pandas-datasets]')
pip('torch', 'torchvision')
pip('segmentation-models-pytorch')
pip('opencv-python-headless')
pip('albumentations')
pip('scikit-learn')
pip('tqdm')
pip('rasterio')           # FIX: was missing from install list
pip('matplotlib')
print('✅ All packages installed.')


In [ ]:
# ── Core imports ─────────────────────────────────────────────────────────────
import os, warnings, random, copy, gc
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from tqdm.notebook import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF

# Segmentation
import segmentation_models_pytorch as smp

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import f1_score, jaccard_score, confusion_matrix

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'🚀 GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('💻 CPU mode — fully functional, slower training')
print(f'🖥️  Device: {DEVICE}')

# ── Global hyperparameters ────────────────────────────────────────────────────
IMG_SIZE    = 256
BATCH_SIZE  = 8 if torch.cuda.is_available() else 4
EPOCHS      = 15
LR          = 1e-4
NUM_WORKERS = 0
N_SYNTHETIC = 200

print(f'\n⚙️  Config: IMG={IMG_SIZE}  BS={BATCH_SIZE}  EP={EPOCHS}  LR={LR}')


## 📊 Section 2 — Dataset Loading (KaggleHub + Robust Fallback)

In [ ]:
# ── Tier 1/2/3 KaggleHub loading strategy ────────────────────────────────────
import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_SLUG = 'akhilchibber/deforestation-detection-dataset'
df = None
dataset_dir = None

print('⬇️  Tier 1: Trying KaggleHub pandas adapter...')
try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET_SLUG, '')
    print(f'✅ Pandas adapter succeeded: {len(df)} rows, cols: {df.columns.tolist()}')
except Exception as e:
    print(f'⚠️  Pandas adapter failed: {e}')

if df is None:
    print('\n⬇️  Tier 2: Trying KaggleHub file download...')
    try:
        dataset_dir = kagglehub.dataset_download(DATASET_SLUG)
        print(f'✅ Files downloaded to: {dataset_dir}')
    except Exception as e:
        print(f'⚠️  File download also failed: {e}')
        print('   → Falling back to Tier 3 (synthetic data)')

print('\n📋 Dataset loading phase complete.')


In [ ]:
# ── Scan for image files ──────────────────────────────────────────────────────
def find_col(keywords, columns):
    for kw in keywords:
        for c in columns:
            if kw in c.lower():
                return c
    return None

IMAGE_COL = LABEL_COL = SAR_COL = None
image_paths = []
HAS_NIR = False

if df is not None:
    cols = df.columns.tolist()
    print(f'📋 Columns: {cols}')
    IMAGE_COL = find_col(['image','img','sentinel2','rgb','path'], cols)
    LABEL_COL = find_col(['label','mask','target','class','deforest'], cols)
    SAR_COL   = find_col(['sar','sentinel1','s1'], cols)
    NIR_COL   = find_col(['nir','b08','b8'], cols)
    HAS_NIR   = NIR_COL is not None
    if IMAGE_COL:
        valid = df[[IMAGE_COL, LABEL_COL]].dropna() if LABEL_COL else df[[IMAGE_COL]].dropna()
        image_paths = list(zip(valid[IMAGE_COL], valid[LABEL_COL])) if LABEL_COL else [(p, None) for p in valid[IMAGE_COL]]
        if image_paths and not Path(str(image_paths[0][0])).exists():
            image_paths = []

if not image_paths and dataset_dir:
    base = Path(dataset_dir)
    img_exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
    all_imgs = sorted([p for p in base.rglob('*') if p.suffix.lower() in img_exts])
    img_files  = [p for p in all_imgs if 'mask' not in p.stem.lower()]
    mask_files = [p for p in all_imgs if 'mask' in p.stem.lower()]
    mask_dict  = {p.stem.replace('_mask','').replace('_label',''): p for p in mask_files}
    for img_p in img_files:
        image_paths.append((img_p, mask_dict.get(img_p.stem)))

USE_SYNTHETIC = len(image_paths) == 0
print(f'\n🔀 Data mode: {"SYNTHETIC (" + str(N_SYNTHETIC) + " samples)" if USE_SYNTHETIC else f"REAL ({len(image_paths)} pairs)"}')


In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
if df is not None:
    print('📐 Shape:', df.shape)
    print('\n📊 Dtypes:\n', df.dtypes.to_string())
    print('\n❓ Null counts:\n', df.isnull().sum().to_string())
    num_cols = df.select_dtypes(include=[np.number]).columns[:8]
    if len(num_cols) > 0:
        fig, axes = plt.subplots(1, min(len(num_cols), 4), figsize=(16, 4))
        if len(num_cols) == 1: axes = [axes]
        for i, col in enumerate(num_cols[:4]):
            axes[i].hist(df[col].dropna(), bins=30, color='teal', edgecolor='black', alpha=0.8)
            axes[i].set_title(col, fontsize=11); axes[i].set_xlabel('Value'); axes[i].set_ylabel('Count')
        plt.suptitle('📊 Numeric Feature Distributions', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()
else:
    print(f'ℹ️  No DataFrame → using {N_SYNTHETIC} synthetic Sentinel-like image pairs.')


## 🧹 Section 3 — Preprocessing, NDVI & Augmentations

In [ ]:
# ── NDVI helpers ──────────────────────────────────────────────────────────────
def compute_ndvi(nir: np.ndarray, red: np.ndarray) -> np.ndarray:
    nir, red = nir.astype(np.float32), red.astype(np.float32)
    denom = nir + red
    ndvi  = np.where(denom == 0, 0.0, (nir - red) / (denom + 1e-8))
    return np.clip(ndvi, -1.0, 1.0).astype(np.float32)

def compute_ndvi_safe(image_hwc: np.ndarray, has_nir: bool = False,
                       nir_ch: int = None, red_ch: int = 0) -> np.ndarray:
    if has_nir and nir_ch is not None:
        return compute_ndvi(image_hwc[:, :, nir_ch], image_hwc[:, :, red_ch])
    return compute_ndvi(image_hwc[:, :, 1], image_hwc[:, :, 0])  # green-proxy

if HAS_NIR:
    print('✅ Real NIR band detected → accurate NDVI will be computed.')
else:
    print('⚠️  No NIR band — using GREEN channel as NIR proxy (standard RGB-only practice).')
    print('   For production accuracy, use Sentinel-2 Band 8 (NIR, 842 nm).')

def normalize_image(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    if img.ndim == 2:
        mn, mx = img.min(), img.max()
        return (img - mn) / (mx - mn + 1e-8)
    out = np.zeros_like(img)
    for c in range(img.shape[-1]):
        mn, mx = img[..., c].min(), img[..., c].max()
        out[..., c] = (img[..., c] - mn) / (mx - mn + 1e-8)
    return out

# ── Augmentation pipelines ────────────────────────────────────────────────────
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    ToTensorV2(),
])

print('\n✅ Preprocessing helpers, NDVI guard, and augmentation pipelines defined.')


In [ ]:
# ── Synthetic Sentinel-like image generator ───────────────────────────────────
def generate_synthetic_pair(size: int = 256, seed: int = None):
    """Returns image HxWx4 (R,G,B,SAR) and mask HxW uint8."""
    rng = np.random.default_rng(seed)
    R   = rng.uniform(0.05, 0.25, (size, size)).astype(np.float32)
    G   = rng.uniform(0.25, 0.55, (size, size)).astype(np.float32)
    B   = rng.uniform(0.05, 0.20, (size, size)).astype(np.float32)
    SAR = rng.uniform(0.10, 0.60, (size, size)).astype(np.float32)
    mask = np.zeros((size, size), dtype=np.uint8)
    for _ in range(rng.integers(2, 6)):
        cx, cy = rng.integers(30, size - 30), rng.integers(30, size - 30)
        rx, ry = rng.integers(15, 60), rng.integers(15, 60)
        Y, X = np.ogrid[:size, :size]
        ellipse = ((X - cx)**2 / rx**2 + (Y - cy)**2 / ry**2) <= 1
        mask[ellipse] = 1
    n_def = mask.sum()
    if n_def > 0:
        R[mask == 1]   = rng.uniform(0.55, 0.80, n_def).astype(np.float32)
        G[mask == 1]   = rng.uniform(0.40, 0.60, n_def).astype(np.float32)
        B[mask == 1]   = rng.uniform(0.20, 0.40, n_def).astype(np.float32)
        SAR[mask == 1] = rng.uniform(0.55, 0.90, n_def).astype(np.float32)
    return np.stack([R, G, B, SAR], axis=-1), mask

# ── Sanity check ──────────────────────────────────────────────────────────────
img_s, msk_s = generate_synthetic_pair(IMG_SIZE, seed=0)
ndvi_preview = compute_ndvi(img_s[:, :, 1], img_s[:, :, 0])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')

axes[0].imshow(img_s[:, :, :3])
axes[0].set_title('🛰️ Synthetic RGB\n(Sentinel-2 proxy)', color='white', fontweight='bold', fontsize=11)
axes[0].axis('off')

im1 = axes[1].imshow(ndvi_preview, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('🌿 NDVI\n(green-proxy)', color='white', fontweight='bold', fontsize=11)
axes[1].axis('off')
cb1 = plt.colorbar(im1, ax=axes[1], fraction=0.046)
cb1.ax.yaxis.set_tick_params(color='white'); cb1.outline.set_edgecolor('white')
plt.setp(cb1.ax.yaxis.get_ticklabels(), color='white')

axes[2].imshow(img_s[:, :, 3], cmap='gray')
axes[2].set_title('📡 SAR Channel\n(Sentinel-1 VV proxy)', color='white', fontweight='bold', fontsize=11)
axes[2].axis('off')

defor_cmap = LinearSegmentedColormap.from_list('defor', ['#1a472a','#e74c3c'])
axes[3].imshow(msk_s, cmap=defor_cmap)
axes[3].set_title(f'🔴 Deforestation Mask\n({msk_s.mean()*100:.1f}% deforested)', color='white', fontweight='bold', fontsize=11)
axes[3].axis('off')
green_patch = mpatches.Patch(color='#1a472a', label='Forest')
red_patch   = mpatches.Patch(color='#e74c3c', label='Deforested')
axes[3].legend(handles=[green_patch, red_patch], loc='lower right', fontsize=8,
               facecolor='#0d1117', labelcolor='white', edgecolor='white')

plt.suptitle('🌍 Synthetic Sentinel-like Sample', fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout(); plt.show()

print(f'Image shape: {img_s.shape}  |  Mask unique: {np.unique(msk_s)}  |  Deforested: {msk_s.mean()*100:.1f}%')


In [ ]:
# ── Build record list ─────────────────────────────────────────────────────────
if USE_SYNTHETIC:
    records = list(range(N_SYNTHETIC))
    print(f'🔧 Using {N_SYNTHETIC} synthetic samples.')
else:
    records = image_paths
    print(f'📂 Using {len(records)} real image-path pairs.')


## 🏗️ Section 4 — PyTorch Dataset & DataLoaders

In [ ]:
# ── DeforestationDataset — FULLY FIXED ───────────────────────────────────────
# FIX 1: _build_multichannel had broken indentation (mixed 2/6 spaces),
#         making the entire method body run at module level (IndentationError)
# FIX 2: __getitem__ was missing entirely
# FIX 3: rasterio was not installed (now added to Section 1)

try:
    import rasterio
    RASTERIO_OK = True
except ImportError:
    RASTERIO_OK = False
    print('⚠️  rasterio not available — file-based loading will use OpenCV fallback')


class DeforestationDataset(Dataset):
    """
    5-channel segmentation dataset.
    Channels: [R, G, B, NDVI, SAR]
    Supports synthetic generation or real image files.
    """

    def __init__(self, records, transform=None,
                 use_synthetic=True, sar_col=None, has_nir=False):
        self.records       = records
        self.transform     = transform
        self.use_synthetic = use_synthetic
        self.sar_col       = sar_col
        self.has_nir       = has_nir

    def __len__(self):
        return len(self.records)

    # ── FIX 2: __getitem__ was completely missing ─────────────────────────────
    def __getitem__(self, idx):
        if self.use_synthetic:
            seed = self.records[idx] if isinstance(self.records[idx], int) else idx
            img_raw, mask = generate_synthetic_pair(IMG_SIZE, seed=seed)
            rgb  = img_raw[:, :, :3]
            sar  = img_raw[:, :, 3]
        else:
            img_path, mask_path = self.records[idx]
            rgb  = self._load_image(str(img_path))
            mask = self._load_mask(str(mask_path)) if mask_path else np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
            sar  = self._load_sar(str(self.sar_col)) if self.sar_col else None

        multichannel = self._build_multichannel(rgb, sar if not self.use_synthetic else sar)
        mask_f = mask.astype(np.float32)

        if self.transform:
            # albumentations expects HWC numpy + mask HW
            aug = self.transform(image=multichannel, mask=mask_f)
            img_t  = aug['image'].float()  # ToTensorV2 → CHW
            mask_t = aug['mask'].unsqueeze(0).float()
        else:
            img_t  = torch.from_numpy(multichannel.transpose(2, 0, 1)).float()
            mask_t = torch.from_numpy(mask_f).unsqueeze(0).float()

        return img_t, mask_t

    # ── Image loader ──────────────────────────────────────────────────────────
    def _load_image(self, path):
        if RASTERIO_OK:
            try:
                with rasterio.open(path) as src:
                    img = np.transpose(src.read(), (1, 2, 0))
            except Exception:
                img = cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        else:
            img = cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if img.shape[2] == 1:
            img = np.repeat(img, 3, axis=2)
        elif img.shape[2] > 3:
            img = img[:, :, :3]
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return normalize_image(img.astype(np.float32))

    # ── Mask loader ───────────────────────────────────────────────────────────
    def _load_mask(self, path):
        mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        return (mask > 127).astype(np.uint8)

    # ── SAR loader ────────────────────────────────────────────────────────────
    def _load_sar(self, path):
        if RASTERIO_OK:
            try:
                with rasterio.open(path) as src:
                    sar = src.read(1)
                sar = cv2.resize(sar, (IMG_SIZE, IMG_SIZE))
                return normalize_image(sar.astype(np.float32))
            except Exception:
                pass
        return None

    # ── FIX 1: _build_multichannel — corrected indentation throughout ─────────
    def _build_multichannel(self, rgb, sar=None):
        # Ensure 3-channel RGB
        if rgb.ndim == 2:
            rgb = np.stack([rgb] * 3, axis=-1)
        if rgb.shape[2] > 3:
            rgb = rgb[:, :, :3]

        # NDVI
        ndvi = compute_ndvi_safe(rgb, has_nir=self.has_nir)
        if ndvi.ndim == 2:
            ndvi = np.expand_dims(ndvi, axis=-1)

        # SAR channel
        if sar is None:
            sar = np.random.uniform(0.1, 0.6, (rgb.shape[0], rgb.shape[1], 1)).astype(np.float32)
        else:
            sar = sar.astype(np.float32)
            if sar.ndim == 2:
                sar = np.expand_dims(sar, axis=-1)
            elif sar.ndim == 3 and sar.shape[2] > 1:
                sar = sar[:, :, :1]

        # Concatenate → [H, W, 5]
        image = np.concatenate([rgb, ndvi, sar], axis=-1).astype(np.float32)

        # Final safety fix for channel count
        c = image.shape[2]
        if c < 5:
            image = np.concatenate([image, np.zeros((image.shape[0], image.shape[1], 5 - c), np.float32)], axis=-1)
        elif c > 5:
            image = image[:, :, :5]

        return image


# ── Quick smoke test ──────────────────────────────────────────────────────────
_test_ds = DeforestationDataset(records=records, transform=None, use_synthetic=USE_SYNTHETIC,
                                 sar_col=SAR_COL, has_nir=HAS_NIR)
_img, _msk = _test_ds[0]
print(f'✅ DeforestationDataset smoke test passed')
print(f'   image tensor: {_img.shape}  dtype: {_img.dtype}')
print(f'   mask  tensor: {_msk.shape}  dtype: {_msk.dtype}')
print(f'   Mask values : {_msk.unique().tolist()}')
del _test_ds


In [ ]:
# ── Build full dataset + DataLoaders ─────────────────────────────────────────
full_dataset = DeforestationDataset(
    records       = records,        # FIX: was records=[] (empty!) in original
    transform     = train_transform,
    use_synthetic = USE_SYNTHETIC,
    sar_col       = SAR_COL,
    has_nir       = HAS_NIR,
)
print(f'✅ Full dataset: {len(full_dataset)} samples')

val_size   = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Validation uses lighter transforms
val_dataset = DeforestationDataset(
    records=records, transform=val_transform,
    use_synthetic=USE_SYNTHETIC, sar_col=SAR_COL, has_nir=HAS_NIR,
)
val_ds.dataset = val_dataset

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# Channel count for model
sample_img, sample_mask = full_dataset[0]
IN_CHANNELS = sample_img.shape[0]

print(f'📦 Train: {len(train_ds)} samples  ({len(train_loader)} batches)')
print(f'📦 Val:   {len(val_ds)} samples  ({len(val_loader)} batches)')
print(f'📊 Input channels: {IN_CHANNELS}')

# ── Visual batch sample ───────────────────────────────────────────────────────
imgs_b, msks_b = next(iter(train_loader))
n_show = min(4, imgs_b.shape[0])

fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 10))
fig.patch.set_facecolor('#111827')
channel_names = ['R', 'G', 'B', 'NDVI', 'SAR']

for i in range(n_show):
    rgb_show = np.clip(imgs_b[i, :3].permute(1,2,0).numpy(), 0, 1)
    axes[0, i].imshow(rgb_show)
    axes[0, i].set_title(f'RGB Sample {i+1}', color='white', fontsize=10, fontweight='bold')
    axes[0, i].axis('off')

    ndvi_show = imgs_b[i, 3].numpy()
    im = axes[1, i].imshow(ndvi_show, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1, i].set_title('NDVI', color='white', fontsize=10, fontweight='bold')
    axes[1, i].axis('off')

    msk_show = msks_b[i, 0].numpy()
    axes[2, i].imshow(msk_show, cmap='hot', vmin=0, vmax=1)
    axes[2, i].set_title(f'Mask ({msk_show.mean()*100:.1f}% defor)', color='white', fontsize=10, fontweight='bold')
    axes[2, i].axis('off')

for ax in axes.flat:
    ax.set_facecolor('#111827')

row_labels = ['RGB Input', 'NDVI Map', 'Ground Truth Mask']
for i, label in enumerate(row_labels):
    fig.text(0.01, 0.84 - i * 0.33, label, va='center', ha='left',
             color='white', fontsize=11, fontweight='bold', rotation=90)

plt.suptitle('📦 Training Batch Sample — Multi-Channel Input', fontsize=14,
             fontweight='bold', color='white', y=1.01)
plt.tight_layout(); plt.show()


## 🧠 Section 5 — U-Net Segmentation Model

In [ ]:
# ── U-Net with ResNet34 encoder ───────────────────────────────────────────────
def build_unet(in_channels: int, encoder: str = 'resnet34') -> nn.Module:
    return smp.Unet(
        encoder_name     = encoder,
        encoder_weights  = 'imagenet',
        in_channels      = in_channels,
        classes          = 1,
        activation       = None,
        decoder_channels = (256, 128, 64, 32, 16),
    )

seg_model = build_unet(IN_CHANNELS).to(DEVICE)

total_p = sum(p.numel() for p in seg_model.parameters())
train_p = sum(p.numel() for p in seg_model.parameters() if p.requires_grad)

print(f'\n✅ U-Net built successfully')
print(f'   Total params     : {total_p:,}')
print(f'   Trainable params : {train_p:,}')
print(f'   Encoder          : ResNet34 (ImageNet pretrained)')
print(f'   In channels      : {IN_CHANNELS}')
print(f'   Output classes   : 1 (binary segmentation)')

# ── Architecture diagram ──────────────────────────────────────────────────────
arch_text = (
    f'Input [{IN_CHANNELS}×{IMG_SIZE}×{IMG_SIZE}]\n'
    '       ↓\n'
    'ResNet34 Encoder\n'
    '  Block1→64  Block2→128  Block3→256  Block4→512\n'
    '       ↓\n'
    'Skip Connections + Decoder\n'
    '  256 → 128 → 64 → 32 → 16\n'
    '       ↓\n'
    'Conv 1×1  →  Sigmoid\n'
    f'Output [1×{IMG_SIZE}×{IMG_SIZE}]  (deforestation prob map)'
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.text(0.5, 0.5, arch_text, transform=ax.transAxes,
        ha='center', va='center', fontsize=13, color='#58a6ff',
        fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=1', facecolor='#161b22', edgecolor='#30363d', linewidth=2))
ax.axis('off')
ax.set_title('🏗️ U-Net Architecture Summary', color='white', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Loss functions ────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        inter   = (probs * targets).sum()
        return 1.0 - (2.0 * inter + self.smooth) / (probs.sum() + targets.sum() + self.smooth)


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce_w, self.dice_w = bce_w, dice_w
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)


criterion = BCEDiceLoss().to(DEVICE)
optimizer = torch.optim.Adam(seg_model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
print('✅ BCE+Dice loss, Adam optimizer, CosineAnnealingLR scheduler ready.')


## 🔁 Section 6 — Training Pipeline + Early Stopping

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_iou(pred, target):
    return float(jaccard_score(target.flatten().astype(int), pred.flatten().astype(int), zero_division=0))

def compute_dice(pred, target):
    return float(f1_score(target.flatten().astype(int), pred.flatten().astype(int), zero_division=0))

def evaluate_batch(logits, masks, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).cpu().numpy().astype(np.uint8)
    gts   = masks.cpu().numpy().astype(np.uint8)
    return (float(np.mean([compute_iou(p.squeeze(), g.squeeze()) for p, g in zip(preds, gts)])),
            float(np.mean([compute_dice(p.squeeze(), g.squeeze()) for p, g in zip(preds, gts)])))

print('✅ Metrics (IoU, Dice/F1) defined.')


In [ ]:
# ── Training + validation loops with early stopping ───────────────────────────
history     = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': [], 'lr': []}
best_val_loss = float('inf')
best_weights  = None
patience, patience_ctr = 5, 0


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    for images, masks in tqdm(loader, desc='  Train', leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward(); optimizer.step()
        total += loss.item() * images.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total, all_iou, all_dice = 0.0, [], []
    for images, masks in tqdm(loader, desc='  Val  ', leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        total += criterion(logits, masks).item() * images.size(0)
        iou, dice = evaluate_batch(logits, masks)
        all_iou.append(iou); all_dice.append(dice)
    return total / len(loader.dataset), float(np.mean(all_iou)), float(np.mean(all_dice))


print('🚀 Starting training...')
print(f'   EPOCHS={EPOCHS}  |  Early stopping patience={patience}  |  Device={DEVICE}')
print('─' * 70)

for epoch in range(1, EPOCHS + 1):
    t_loss               = train_one_epoch(seg_model, train_loader, criterion, optimizer)
    v_loss, v_iou, v_dice = validate(seg_model, val_loader, criterion)
    current_lr           = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_iou'].append(v_iou)
    history['val_dice'].append(v_dice)
    history['lr'].append(current_lr)

    improved = v_loss < best_val_loss
    if improved:
        best_val_loss = v_loss
        best_weights  = copy.deepcopy(seg_model.state_dict())
        patience_ctr  = 0; tag = '✅ best'
    else:
        patience_ctr += 1; tag = f'⏳ ({patience_ctr}/{patience})'

    print(f'Ep [{epoch:02d}/{EPOCHS}]  TrLoss={t_loss:.4f}  ValLoss={v_loss:.4f}  '
          f'IoU={v_iou:.4f}  Dice={v_dice:.4f}  LR={current_lr:.2e}  {tag}')

    if patience_ctr >= patience:
        print(f'\n⛔ Early stopping at epoch {epoch}.')
        break

seg_model.load_state_dict(best_weights)
print(f'\n✅ Training complete.  Best val loss: {best_val_loss:.4f}')


In [ ]:
# ── Enhanced Training Curves Dashboard ────────────────────────────────────────
ep = list(range(1, len(history['train_loss']) + 1))

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#0d1117')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax_cfg = dict(facecolor='#161b22')

# Loss curves
ax0 = fig.add_subplot(gs[0, :2]); ax0.set(**ax_cfg)
ax0.plot(ep, history['train_loss'], 'o-', color='#58a6ff', linewidth=2.5, markersize=6, label='Train Loss')
ax0.plot(ep, history['val_loss'],   's-', color='#f85149', linewidth=2.5, markersize=6, label='Val Loss')
ax0.fill_between(ep, history['train_loss'], history['val_loss'],
                  alpha=0.12, color='#a371f7', label='Gap')
best_ep = ep[np.argmin(history['val_loss'])]
ax0.axvline(best_ep, color='#3fb950', linestyle='--', alpha=0.8, linewidth=1.5, label=f'Best epoch ({best_ep})')
ax0.set_title('📉 Loss Curves (Train vs Val)', color='white', fontsize=13, fontweight='bold')
ax0.set_xlabel('Epoch', color='#8b949e'); ax0.set_ylabel('Loss', color='#8b949e')
ax0.tick_params(colors='#8b949e'); ax0.spines[:].set_color('#30363d')
ax0.legend(facecolor='#161b22', labelcolor='white', edgecolor='#30363d')
ax0.grid(alpha=0.2, color='#30363d')

# LR schedule
ax1 = fig.add_subplot(gs[0, 2]); ax1.set(**ax_cfg)
ax1.semilogy(ep, history['lr'], '^-', color='#ffa657', linewidth=2, markersize=5)
ax1.set_title('📐 Learning Rate Schedule', color='white', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch', color='#8b949e'); ax1.set_ylabel('LR (log)', color='#8b949e')
ax1.tick_params(colors='#8b949e'); ax1.spines[:].set_color('#30363d')
ax1.grid(alpha=0.2, color='#30363d')

# IoU
ax2 = fig.add_subplot(gs[1, 0]); ax2.set(**ax_cfg)
ax2.plot(ep, history['val_iou'], 'D-', color='#3fb950', linewidth=2.5, markersize=7)
ax2.fill_between(ep, 0, history['val_iou'], alpha=0.2, color='#3fb950')
ax2.set_title('📏 Validation IoU (Jaccard)', color='white', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch', color='#8b949e'); ax2.set_ylabel('IoU', color='#8b949e')
ax2.set_ylim(0, 1); ax2.tick_params(colors='#8b949e'); ax2.spines[:].set_color('#30363d')
ax2.grid(alpha=0.2, color='#30363d')

# Dice
ax3 = fig.add_subplot(gs[1, 1]); ax3.set(**ax_cfg)
ax3.plot(ep, history['val_dice'], 'v-', color='#a371f7', linewidth=2.5, markersize=7)
ax3.fill_between(ep, 0, history['val_dice'], alpha=0.2, color='#a371f7')
ax3.set_title('🎯 Validation Dice / F1', color='white', fontsize=13, fontweight='bold')
ax3.set_xlabel('Epoch', color='#8b949e'); ax3.set_ylabel('Dice', color='#8b949e')
ax3.set_ylim(0, 1); ax3.tick_params(colors='#8b949e'); ax3.spines[:].set_color('#30363d')
ax3.grid(alpha=0.2, color='#30363d')

# Summary box
ax4 = fig.add_subplot(gs[1, 2]); ax4.set(**ax_cfg); ax4.axis('off')
best_iou  = max(history['val_iou'])
best_dice = max(history['val_dice'])
summary = (
    f'  Training Summary\n'
    f'  ─────────────────────\n'
    f'  Epochs run:    {len(ep)}\n'
    f'  Best Val Loss: {best_val_loss:.4f}\n'
    f'  Best IoU:      {best_iou:.4f}\n'
    f'  Best Dice/F1:  {best_dice:.4f}\n'
    f'  Best Epoch:    {best_ep}\n'
    f'  Device:        {DEVICE}'
)
ax4.text(0.05, 0.5, summary, transform=ax4.transAxes, va='center', ha='left',
         fontsize=12, color='#e6edf3', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#21262d', edgecolor='#3fb950', linewidth=2))

plt.suptitle('🌳 Training History — Deforestation Segmentation', fontsize=16,
             fontweight='bold', color='white', y=1.01)
plt.show()


In [ ]:
# ── Enhanced Prediction Gallery ────────────────────────────────────────────────
@torch.no_grad()
def show_predictions(model, dataset, n=4, threshold=0.5):
    model.eval()
    defor_cmap = LinearSegmentedColormap.from_list('defor', ['#1a472a','#e74c3c'])
    prob_cmap  = LinearSegmentedColormap.from_list('prob',  ['#0d1117','#ffa657','#e74c3c'])

    fig, axes = plt.subplots(n, 5, figsize=(20, 4.5 * n))
    fig.patch.set_facecolor('#0d1117')
    col_titles = ['🛰️ RGB Input', '🌿 NDVI', '✅ Ground Truth', '🔥 Prob Map', '🔴 Prediction + IoU']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, color='white', fontsize=11, fontweight='bold', pad=8)

    for i in range(n):
        img_t, msk_t = dataset[i]
        logit = model(img_t.unsqueeze(0).to(DEVICE))
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()
        pred  = (prob > threshold).astype(np.uint8)
        iou   = compute_iou(pred, msk_t.squeeze().numpy())
        dice  = compute_dice(pred, msk_t.squeeze().numpy())

        rgb  = np.clip(img_t[:3].permute(1,2,0).cpu().numpy(), 0, 1)
        ndvi = img_t[3].cpu().numpy()
        gt   = msk_t.squeeze().numpy()

        axes[i,0].imshow(rgb);                      axes[i,0].axis('off')
        im_n = axes[i,1].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1); axes[i,1].axis('off')
        plt.colorbar(im_n, ax=axes[i,1], fraction=0.046)
        axes[i,2].imshow(gt, cmap=defor_cmap, vmin=0, vmax=1); axes[i,2].axis('off')
        im_p = axes[i,3].imshow(prob, cmap=prob_cmap, vmin=0, vmax=1); axes[i,3].axis('off')
        plt.colorbar(im_p, ax=axes[i,3], fraction=0.046)
        axes[i,4].imshow(pred, cmap=defor_cmap, vmin=0, vmax=1)
        axes[i,4].set_xlabel(f'IoU={iou:.3f}  Dice={dice:.3f}', color='#3fb950', fontsize=10)
        axes[i,4].axis('off')

        for j in range(5):
            axes[i,j].set_facecolor('#0d1117')

    plt.suptitle('🔍 U-Net Deforestation Predictions', fontsize=16, fontweight='bold', color='white', y=1.01)
    plt.tight_layout(); plt.show()

show_predictions(seg_model, full_dataset, n=4)


## 🔍 Section 7 — Change Detection (NDVI Differencing)

In [ ]:
# ── Bi-temporal change detection ──────────────────────────────────────────────
def detect_change_ndvi(img_before, img_after, threshold=0.10):
    ndvi_b = compute_ndvi(img_before[:,:,1], img_before[:,:,0])
    ndvi_a = compute_ndvi(img_after[:,:,1],  img_after[:,:,0])
    diff   = ndvi_b - ndvi_a
    return diff.astype(np.float32), (diff > threshold).astype(np.uint8)

def detect_change_image_diff(img_before, img_after, threshold=0.15):
    diff_map = np.abs(img_before.astype(float) - img_after.astype(float)).mean(axis=-1).astype(np.float32)
    return diff_map, (diff_map > threshold).astype(np.uint8)

# Two temporal frames
img_t1, mask_t1 = generate_synthetic_pair(IMG_SIZE, seed=10)
img_t2, mask_t2 = generate_synthetic_pair(IMG_SIZE, seed=99)

ndvi_diff, ndvi_change = detect_change_ndvi(img_t1, img_t2, threshold=0.10)
img_diff,  img_change  = detect_change_image_diff(img_t1[:,:,:3], img_t2[:,:,:3], threshold=0.15)

defor_cmap = LinearSegmentedColormap.from_list('defor', ['#1a472a','#e74c3c'])
diff_cmap  = LinearSegmentedColormap.from_list('diff',  ['#1565c0','#f5f5f5','#b71c1c'])

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.patch.set_facecolor('#0d1117')
for ax in axes.flat: ax.set_facecolor('#0d1117')

# Row 0: T1
ndvi_t1 = compute_ndvi(img_t1[:,:,1], img_t1[:,:,0])
axes[0,0].imshow(img_t1[:,:,:3]);                     axes[0,0].set_title('🛰️ T1 — Before (RGB)',  color='white', fontweight='bold'); axes[0,0].axis('off')
im = axes[0,1].imshow(ndvi_t1, cmap='RdYlGn', vmin=-1, vmax=1); axes[0,1].set_title('🌿 T1 — NDVI', color='white', fontweight='bold'); axes[0,1].axis('off'); plt.colorbar(im, ax=axes[0,1], fraction=0.046)
axes[0,2].imshow(mask_t1, cmap=defor_cmap);           axes[0,2].set_title('🔴 T1 — GT Mask',       color='white', fontweight='bold'); axes[0,2].axis('off')
axes[0,3].imshow(img_t1[:,:,3], cmap='gray');          axes[0,3].set_title('📡 T1 — SAR',            color='white', fontweight='bold'); axes[0,3].axis('off')

# Row 1: T2
ndvi_t2 = compute_ndvi(img_t2[:,:,1], img_t2[:,:,0])
axes[1,0].imshow(img_t2[:,:,:3]);                     axes[1,0].set_title('🛰️ T2 — After (RGB)',  color='white', fontweight='bold'); axes[1,0].axis('off')
im2 = axes[1,1].imshow(ndvi_t2, cmap='RdYlGn', vmin=-1, vmax=1); axes[1,1].set_title('🌿 T2 — NDVI', color='white', fontweight='bold'); axes[1,1].axis('off'); plt.colorbar(im2, ax=axes[1,1], fraction=0.046)
axes[1,2].imshow(mask_t2, cmap=defor_cmap);           axes[1,2].set_title('🔴 T2 — GT Mask',      color='white', fontweight='bold'); axes[1,2].axis('off')
axes[1,3].imshow(img_t2[:,:,3], cmap='gray');          axes[1,3].set_title('📡 T2 — SAR',           color='white', fontweight='bold'); axes[1,3].axis('off')

# Row 2: Change maps
im_d = axes[2,0].imshow(ndvi_diff, cmap=diff_cmap, vmin=-0.5, vmax=0.5)
axes[2,0].set_title('🔄 NDVI Difference (T1−T2)\n(red = vegetation loss)', color='white', fontweight='bold'); axes[2,0].axis('off'); plt.colorbar(im_d, ax=axes[2,0], fraction=0.046)

axes[2,1].imshow(ndvi_change, cmap=defor_cmap)
axes[2,1].set_title(f'🚨 NDVI Change Map\n({ndvi_change.mean()*100:.1f}% changed)', color='white', fontweight='bold'); axes[2,1].axis('off')

im_id = axes[2,2].imshow(img_diff, cmap='inferno')
axes[2,2].set_title('🔄 Image Diff (pixel)', color='white', fontweight='bold'); axes[2,2].axis('off'); plt.colorbar(im_id, ax=axes[2,2], fraction=0.046)

axes[2,3].imshow(img_change, cmap=defor_cmap)
axes[2,3].set_title(f'🚨 Image Change Map\n({img_change.mean()*100:.1f}% changed)', color='white', fontweight='bold'); axes[2,3].axis('off')

plt.suptitle('🔍 Bi-Temporal Change Detection — NDVI Differencing', fontsize=16,
             fontweight='bold', color='white', y=1.01)
plt.tight_layout(); plt.show()

print(f'NDVI method   → {ndvi_change.mean()*100:.1f}% changed')
print(f'Image method  → {img_change.mean()*100:.1f}% changed')


## 🟣 Section 8 — Temporal Model (CNN + LSTM)

In [ ]:
print('ℹ️  Temporal model uses SYNTHETIC sequences.')
print('   Real implementation: use Sentinel-2 monthly composites.')

class CNNFeatureExtractor(nn.Module):
    def __init__(self, in_channels=5, feature_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),          nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),         nn.BatchNorm2d(128), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
        )
        self.fc = nn.Linear(128 * 16, feature_dim)

    def forward(self, x):
        return F.relu(self.fc(self.encoder(x)))


class TemporalDeforestationModel(nn.Module):
    """CNN + LSTM for temporal deforestation. Input: (B, T, C, H, W) → Output: (B, T, 1)"""
    def __init__(self, in_channels=5, feature_dim=128, hidden_dim=64, num_layers=2):
        super().__init__()
        self.cnn  = CNNFeatureExtractor(in_channels, feature_dim)
        self.lstm = nn.LSTM(feature_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        feats = torch.stack([self.cnn(x[:, t]) for t in range(T)], dim=1)
        out, _ = self.lstm(feats)
        return torch.sigmoid(self.head(out))


temporal_model = TemporalDeforestationModel(
    in_channels=IN_CHANNELS, feature_dim=128, hidden_dim=64, num_layers=2
).to(DEVICE)
print(f'✅ Temporal model: {sum(p.numel() for p in temporal_model.parameters()):,} parameters')


In [ ]:
# ── Synthetic temporal dataset + training ─────────────────────────────────────
class TemporalSequenceDataset(Dataset):
    def __init__(self, n=100, seq_len=6, img_size=64, in_channels=5):
        self.n, self.T, self.size, self.C = n, seq_len, img_size, in_channels

    def __len__(self): return self.n

    def __getitem__(self, idx):
        frames, labels = [], []
        for t in range(self.T):
            img, msk = generate_synthetic_pair(self.size, seed=idx * self.T + t)
            rgb  = img[:, :, :3]
            sar  = img[:, :, 3:4]
            ndvi = compute_ndvi(img[:,:,1], img[:,:,0])[:,:,np.newaxis]
            frame = np.concatenate([rgb, ndvi, sar], axis=-1).transpose(2,0,1)
            frames.append(torch.from_numpy(frame).float())
            labels.append(float(msk.mean()))
        return torch.stack(frames), torch.tensor(labels).float().unsqueeze(-1)

SEQ_LEN  = 6
temp_ds  = TemporalSequenceDataset(n=120, seq_len=SEQ_LEN, img_size=64, in_channels=IN_CHANNELS)
temp_ldr = DataLoader(temp_ds, batch_size=4, shuffle=True, num_workers=0)

xb, yb = next(iter(temp_ldr))
print(f'Sequence batch: {xb.shape}  labels: {yb.shape}')

temp_opt  = torch.optim.Adam(temporal_model.parameters(), lr=1e-3)
temp_crit = nn.MSELoss()
TEMP_EPOCHS = 8
temp_losses = []

print('\n🚀 Training temporal CNN+LSTM...')
for ep_t in range(1, TEMP_EPOCHS + 1):
    temporal_model.train()
    ep_loss = 0.0
    for seqs, lbls in temp_ldr:
        seqs, lbls = seqs.to(DEVICE), lbls.to(DEVICE)
        temp_opt.zero_grad()
        loss = temp_crit(temporal_model(seqs), lbls)
        loss.backward(); temp_opt.step()
        ep_loss += loss.item()
    ep_loss /= len(temp_ldr)
    temp_losses.append(ep_loss)
    print(f'  Epoch [{ep_t}/{TEMP_EPOCHS}]  MSE: {ep_loss:.5f}')
print('✅ Temporal model trained.')

# Training loss curve
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
ax.plot(range(1, TEMP_EPOCHS+1), temp_losses, 'o-', color='#a371f7', linewidth=2.5, markersize=7)
ax.fill_between(range(1, TEMP_EPOCHS+1), 0, temp_losses, alpha=0.2, color='#a371f7')
ax.set_title('🟣 Temporal CNN+LSTM Training Loss', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', color='#8b949e'); ax.set_ylabel('MSE Loss', color='#8b949e')
ax.tick_params(colors='#8b949e'); ax.spines[:].set_color('#30363d')
ax.grid(alpha=0.2, color='#30363d'); plt.tight_layout(); plt.show()


In [ ]:
# ── Enhanced temporal predictions visualisation ───────────────────────────────
@torch.no_grad()
def plot_temporal_predictions(model, dataset, idx=0):
    model.eval()
    seqs, lbls = dataset[idx]
    preds = model(seqs.unsqueeze(0).to(DEVICE)).squeeze().cpu().numpy()
    trues = lbls.squeeze().numpy()
    T = len(trues)

    fig = plt.figure(figsize=(4*T, 12))
    fig.patch.set_facecolor('#0d1117')
    gs_inner = gridspec.GridSpec(2, T, figure=fig, hspace=0.1, wspace=0.05, top=0.88, bottom=0.38)
    gs_plot  = gridspec.GridSpec(1, 1, figure=fig, top=0.30, bottom=0.05)

    for t in range(T):
        ax = fig.add_subplot(gs_inner[0, t])
        ax.set_facecolor('#0d1117')
        rgb = np.clip(seqs[t, :3].permute(1,2,0).numpy(), 0, 1)
        ax.imshow(rgb); ax.axis('off')
        ax.set_title(f'T={t+1}', color='white', fontsize=10)

        ax2 = fig.add_subplot(gs_inner[1, t])
        ax2.set_facecolor('#0d1117')
        ndvi = seqs[t, 3].numpy()
        ax2.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1); ax2.axis('off')
        if t == 0: ax2.set_ylabel('NDVI', color='white', fontsize=9)

    ax_plot = fig.add_subplot(gs_plot[0])
    ax_plot.set_facecolor('#161b22')
    ax_plot.plot(range(1, T+1), trues, 'o-', color='#3fb950', linewidth=2.5, markersize=8, label='True deforestation fraction')
    ax_plot.plot(range(1, T+1), preds, 's--', color='#f85149', linewidth=2.5, markersize=8, label='LSTM prediction')
    ax_plot.fill_between(range(1, T+1), trues, preds, alpha=0.15, color='#ffa657')
    ax_plot.set_xlabel('Time Step', color='#8b949e', fontsize=11)
    ax_plot.set_ylabel('Deforestation Fraction', color='#8b949e', fontsize=11)
    ax_plot.set_title('CNN + LSTM Temporal Prediction vs Ground Truth', color='white', fontsize=12, fontweight='bold')
    ax_plot.legend(facecolor='#161b22', labelcolor='white', edgecolor='#30363d', fontsize=10)
    ax_plot.tick_params(colors='#8b949e'); ax_plot.spines[:].set_color('#30363d')
    ax_plot.grid(alpha=0.2, color='#30363d'); ax_plot.set_ylim(0, 0.5)

    plt.suptitle('🟣 CNN + LSTM Temporal Deforestation Model', fontsize=15, fontweight='bold', color='white')
    plt.show()

plot_temporal_predictions(temporal_model, temp_ds, idx=0)


## 🔮 Section 9 — Future Deforestation Risk Forecasting

In [ ]:
class DeforestationForecaster:
    def __init__(self, temporal_model, seg_model, device):
        self.temporal = temporal_model
        self.seg      = seg_model
        self.device   = device

    @torch.no_grad()
    def predict_future(self, sequence: torch.Tensor, n_future: int = 4) -> np.ndarray:
        self.temporal.eval()
        probs = self.temporal(sequence.unsqueeze(0).to(self.device)).squeeze().cpu().numpy()
        if len(probs) >= 3:
            from numpy.polynomial import polynomial as P
            x = np.arange(len(probs))
            c = P.polyfit(x, probs, deg=1)
            future = np.clip(P.polyval(np.arange(len(probs), len(probs) + n_future), c), 0, 1)
        else:
            future = np.full(n_future, probs[-1])
        return np.concatenate([probs, future])

    def risk_category(self, prob: float) -> tuple:
        if prob < 0.20:   return '🟢 LOW',      '#3fb950'
        elif prob < 0.50: return '🟡 MODERATE',  '#ffa657'
        elif prob < 0.75: return '🟠 HIGH',       '#f0883e'
        else:             return '🔴 CRITICAL',   '#f85149'


forecaster  = DeforestationForecaster(temporal_model, seg_model, DEVICE)
seq_demo, _ = temp_ds[5]
N_FUTURE    = 4
all_probs   = forecaster.predict_future(seq_demo, n_future=N_FUTURE)

n_obs        = SEQ_LEN
obs_steps    = list(range(1, n_obs + 1))
future_steps = list(range(n_obs + 1, n_obs + N_FUTURE + 1))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

# ── Main forecast plot ────────────────────────────────────────────────────────
ax = axes[0]; ax.set_facecolor('#161b22')
ax.plot(obs_steps, all_probs[:n_obs], 'o-', color='#58a6ff', linewidth=2.5, markersize=9, label='Observed', zorder=5)
ax.plot(future_steps, all_probs[n_obs:], 's--', color='#f85149', linewidth=2.5, markersize=9, label='Forecast', zorder=5)
ax.axvline(x=n_obs + 0.5, color='#8b949e', linestyle=':', linewidth=1.5, label='Forecast start')
ax.fill_between(future_steps,
                np.clip(all_probs[n_obs:] - 0.08, 0, 1),
                np.clip(all_probs[n_obs:] + 0.08, 0, 1),
                alpha=0.25, color='#f85149', label='Uncertainty band')
ax.axhline(y=0.20, color='#3fb950', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=0.50, color='#ffa657', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=0.75, color='#f0883e', linestyle='--', alpha=0.5, linewidth=1)
ax.text(n_obs + N_FUTURE + 0.05, 0.10, 'LOW',      color='#3fb950', fontsize=9, va='center')
ax.text(n_obs + N_FUTURE + 0.05, 0.35, 'MODERATE', color='#ffa657', fontsize=9, va='center')
ax.text(n_obs + N_FUTURE + 0.05, 0.63, 'HIGH',     color='#f0883e', fontsize=9, va='center')
ax.text(n_obs + N_FUTURE + 0.05, 0.88, 'CRITICAL', color='#f85149', fontsize=9, va='center')
ax.set_xlabel('Time Step (months)', color='#8b949e', fontsize=12)
ax.set_ylabel('Deforestation Probability', color='#8b949e', fontsize=12)
ax.set_title('🔮 Future Deforestation Risk Forecast', color='white', fontsize=13, fontweight='bold')
ax.legend(facecolor='#161b22', labelcolor='white', edgecolor='#30363d', fontsize=10)
ax.tick_params(colors='#8b949e'); ax.spines[:].set_color('#30363d')
ax.grid(alpha=0.2, color='#30363d'); ax.set_ylim(0, 1.05)

# ── Risk bar chart ────────────────────────────────────────────────────────────
ax2 = axes[1]; ax2.set_facecolor('#161b22')
future_labels = [f'T+{i}' for i in range(1, N_FUTURE+1)]
colors = [forecaster.risk_category(p)[1] for p in all_probs[n_obs:]]
bars = ax2.bar(future_labels, all_probs[n_obs:], color=colors, edgecolor='#30363d', linewidth=1.5, width=0.6)
for bar, prob in zip(bars, all_probs[n_obs:]):
    label, _ = forecaster.risk_category(prob)
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{prob:.3f}\n{label}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax2.set_ylim(0, 1.2); ax2.set_xlabel('Future Month', color='#8b949e', fontsize=12)
ax2.set_ylabel('Deforestation Probability', color='#8b949e', fontsize=12)
ax2.set_title('📊 Risk Category per Future Month', color='white', fontsize=13, fontweight='bold')
ax2.tick_params(colors='#8b949e'); ax2.spines[:].set_color('#30363d')
ax2.grid(alpha=0.2, color='#30363d', axis='y')

plt.suptitle('🌍 Deforestation Risk Forecasting System', fontsize=16, fontweight='bold', color='white', y=1.01)
plt.tight_layout(); plt.show()

print('\n📋 Risk Assessment:')
for i, p in enumerate(all_probs[n_obs:], 1):
    label, _ = forecaster.risk_category(p)
    print(f'   Month T+{i}: prob={p:.3f}  →  {label}')


## ⚡ Section 10 — Model Compression (FP16 + Pruning)

In [ ]:
import torch.nn.utils.prune as prune

# FIX: compress_model returns a tuple — must unpack correctly
def compress_model(model: nn.Module, prune_amount: float = 0.20):
    """Returns (compressed_model, sparsity_fraction)."""
    compressed = copy.deepcopy(model).cpu()
    for name, module in compressed.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.l1_unstructured(module, name='weight', amount=prune_amount)
            prune.remove(module, 'weight')
    total  = sum(p.numel() for p in compressed.parameters())
    zeroed = sum((p == 0).sum().item() for p in compressed.parameters())
    return compressed, zeroed / total


# FIX: properly unpack the returned tuple
compressed_model, sparsity = compress_model(seg_model)

orig_size = sum(p.numel() * 4 for p in seg_model.parameters()) / 1e6   # FP32 MB
comp_size = sum(p.numel() * 4 for p in compressed_model.parameters()) / 1e6

print(f'✅ Model compressed')
print(f'   Pruning:         {sparsity*100:.1f}% of weights zeroed (L1 magnitude)')
print(f'   Original size:   ~{orig_size:.1f} MB (FP32)')
print(f'   Effective size:  ~{orig_size*(1-sparsity):.1f} MB (after pruning)')

# FP16 test
fp16_model = copy.deepcopy(seg_model).half().to(DEVICE)
dummy = torch.randn(1, IN_CHANNELS, IMG_SIZE, IMG_SIZE).half().to(DEVICE)
with torch.no_grad():
    out = fp16_model(dummy)
print(f'   FP16 inference output shape: {out.shape}')
print(f'   Memory: ~50% vs FP32')
del fp16_model, dummy; gc.collect()

# Visual summary
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
labels   = ['Original\nFP32', 'After Pruning\n(FP32 equiv)', 'FP16\n(half precision)']
sizes    = [orig_size, orig_size * (1-sparsity), orig_size * 0.5]
colors   = ['#58a6ff', '#ffa657', '#3fb950']
bars     = ax.bar(labels, sizes, color=colors, edgecolor='#30363d', width=0.5, linewidth=1.5)
for bar, val in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f} MB', ha='center', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel('Model Size (MB)', color='#8b949e', fontsize=11)
ax.set_title('⚡ Model Compression Summary', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='#8b949e'); ax.spines[:].set_color('#30363d')
ax.grid(alpha=0.2, color='#30363d', axis='y'); ax.set_ylim(0, max(sizes) * 1.3)
plt.tight_layout(); plt.show()


## 📊 Section 11 — Full Visualisation Dashboard

In [ ]:
# ── Enhanced results dashboard ────────────────────────────────────────────────
@torch.no_grad()
def deforestation_dashboard(seg_model, dataset, n_samples=3, threshold=0.5):
    seg_model.eval()
    defor_cmap = LinearSegmentedColormap.from_list('defor', ['#1a472a','#e74c3c'])
    prob_cmap  = LinearSegmentedColormap.from_list('prob',  ['#0d1117','#ffa657','#e74c3c'])

    fig = plt.figure(figsize=(22, 7 * n_samples + 3))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(n_samples + 1, 6, figure=fig, hspace=0.45, wspace=0.3)

    all_ious, all_dices = [], []

    for i in range(n_samples):
        img_t, msk_t = dataset[i]
        logit = seg_model(img_t.unsqueeze(0).to(DEVICE))
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()
        pred  = (prob > threshold).astype(np.uint8)
        gt    = msk_t.squeeze().numpy().astype(np.uint8)

        iou  = compute_iou(pred, gt)
        dice = compute_dice(pred, gt)
        all_ious.append(iou); all_dices.append(dice)

        rgb  = np.clip(img_t[:3].permute(1,2,0).cpu().numpy(), 0, 1)
        ndvi = img_t[3].cpu().numpy()
        sar  = img_t[4].cpu().numpy()

        ax_rgb  = fig.add_subplot(gs[i, 0]); ax_rgb.imshow(rgb); ax_rgb.set_title(f'🛰️ RGB [{i+1}]', color='white', fontweight='bold', fontsize=10); ax_rgb.axis('off')
        ax_ndvi = fig.add_subplot(gs[i, 1]); im_n = ax_ndvi.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1); ax_ndvi.set_title('🌿 NDVI', color='white', fontweight='bold', fontsize=10); ax_ndvi.axis('off'); plt.colorbar(im_n, ax=ax_ndvi, fraction=0.046)
        ax_sar  = fig.add_subplot(gs[i, 2]); ax_sar.imshow(sar, cmap='gray'); ax_sar.set_title('📡 SAR', color='white', fontweight='bold', fontsize=10); ax_sar.axis('off')
        ax_gt   = fig.add_subplot(gs[i, 3]); ax_gt.imshow(gt, cmap=defor_cmap, vmin=0, vmax=1); ax_gt.set_title('✅ Ground Truth', color='white', fontweight='bold', fontsize=10); ax_gt.axis('off')
        ax_prob = fig.add_subplot(gs[i, 4]); im_p = ax_prob.imshow(prob, cmap=prob_cmap, vmin=0, vmax=1); ax_prob.set_title('🔥 Prob Map', color='white', fontweight='bold', fontsize=10); ax_prob.axis('off'); plt.colorbar(im_p, ax=ax_prob, fraction=0.046)
        ax_pred = fig.add_subplot(gs[i, 5]); ax_pred.imshow(pred, cmap=defor_cmap, vmin=0, vmax=1)
        ax_pred.set_title(f'🔴 Prediction\nIoU={iou:.3f}  Dice={dice:.3f}', color='#3fb950' if iou > 0.5 else '#ffa657', fontweight='bold', fontsize=10); ax_pred.axis('off')

        for ax in [ax_rgb, ax_ndvi, ax_sar, ax_gt, ax_prob, ax_pred]:
            ax.set_facecolor('#0d1117')

    # Summary row
    ax_sum = fig.add_subplot(gs[n_samples, :])
    ax_sum.set_facecolor('#161b22'); ax_sum.axis('off')
    summary = (
        f'📊 Dashboard Summary  |  Samples: {n_samples}  |  '
        f'Mean IoU: {np.mean(all_ious):.4f}  |  Mean Dice: {np.mean(all_dices):.4f}  |  '
        f'Best IoU: {max(all_ious):.4f}  |  Device: {DEVICE}'
    )
    ax_sum.text(0.5, 0.5, summary, transform=ax_sum.transAxes,
                ha='center', va='center', color='#e6edf3', fontsize=12,
                fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.6', facecolor='#21262d', edgecolor='#3fb950', linewidth=2))

    plt.suptitle('🌳 Deforestation Detection Dashboard — Full Pipeline Results',
                 fontsize=16, fontweight='bold', color='white', y=1.01)
    plt.show()

deforestation_dashboard(seg_model, full_dataset, n_samples=3)


## 🔬 Section 12 — Production Inference Function

In [ ]:
@torch.no_grad()
def predict_deforestation(input_data, model=None, threshold=0.5, show_plot=True) -> dict:
    """
    Unified inference. Accepts: torch.Tensor (C,H,W), np.ndarray (H,W,C), or file path.
    Returns: {'prob_map', 'binary_mask', 'deforestation_pct'}
    """
    if model is None:
        model = seg_model
    model.eval()

    if isinstance(input_data, (str, Path)):
        raw = cv2.imread(str(input_data), cv2.IMREAD_UNCHANGED)
        if raw is None: raise FileNotFoundError(f'Cannot load: {input_data}')
        raw = cv2.cvtColor(raw[:,:,:3], cv2.COLOR_BGR2RGB)
        raw = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
        img_np = normalize_image(raw.astype(np.float32))
        ndvi   = compute_ndvi_safe(img_np, has_nir=HAS_NIR)[:,:,np.newaxis]
        sar    = np.random.uniform(0.1, 0.6, (IMG_SIZE, IMG_SIZE, 1)).astype(np.float32)
        img_np = np.concatenate([img_np, ndvi, sar], axis=-1)
    elif isinstance(input_data, np.ndarray):
        img_np = input_data.astype(np.float32)
        if img_np.ndim == 2:
            img_np = np.stack([img_np]*3 + [np.zeros_like(img_np)]*2, axis=-1)
    elif isinstance(input_data, torch.Tensor):
        img_np = input_data.cpu().numpy()
        if img_np.ndim == 3: img_np = img_np.transpose(1, 2, 0)
    else:
        raise TypeError(f'Unsupported input type: {type(input_data)}')

    # Pad/trim to IN_CHANNELS
    if img_np.shape[-1] < IN_CHANNELS:
        pad = IN_CHANNELS - img_np.shape[-1]
        img_np = np.concatenate([img_np, np.zeros((*img_np.shape[:2], pad), np.float32)], axis=-1)

    img_tensor = torch.from_numpy(img_np[:,:,:IN_CHANNELS].transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
    prob   = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
    binary = (prob > threshold).astype(np.uint8)
    pct    = float(binary.mean() * 100)

    if show_plot:
        rgb_disp = np.clip(img_np[:,:,:3], 0, 1)
        defor_cmap = LinearSegmentedColormap.from_list('defor', ['#1a472a','#e74c3c'])
        prob_cmap  = LinearSegmentedColormap.from_list('prob',  ['#0d1117','#ffa657','#e74c3c'])

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.patch.set_facecolor('#0d1117')
        for ax in axes: ax.set_facecolor('#0d1117')

        axes[0].imshow(rgb_disp); axes[0].set_title('🛰️ Input RGB', color='white', fontweight='bold'); axes[0].axis('off')

        ndvi_vis = compute_ndvi_safe(img_np[:,:,:3], has_nir=False)
        im_n = axes[1].imshow(ndvi_vis, cmap='RdYlGn', vmin=-1, vmax=1)
        axes[1].set_title('🌿 NDVI', color='white', fontweight='bold'); axes[1].axis('off')
        plt.colorbar(im_n, ax=axes[1], fraction=0.046)

        im_p = axes[2].imshow(prob, cmap=prob_cmap, vmin=0, vmax=1)
        axes[2].set_title('🔥 Deforestation Probability', color='white', fontweight='bold'); axes[2].axis('off')
        plt.colorbar(im_p, ax=axes[2], fraction=0.046)

        axes[3].imshow(binary, cmap=defor_cmap, vmin=0, vmax=1)
        axes[3].set_title(f'🔴 Binary Mask\n{pct:.1f}% deforested', color='white', fontweight='bold'); axes[3].axis('off')

        plt.suptitle('🌳 Deforestation Detection — Production Inference', fontsize=14, fontweight='bold', color='white')
        plt.tight_layout(); plt.show()

    return {'prob_map': prob, 'binary_mask': binary, 'deforestation_pct': pct}


print('🔍 Running inference on sample from dataset...')
sample_tensor, _ = full_dataset[7]
result = predict_deforestation(sample_tensor, model=seg_model, threshold=0.5, show_plot=True)
print(f"\n📊 Result: {result['deforestation_pct']:.2f}% of image area deforested")
print(f"   Prob map:    {result['prob_map'].shape}   min={result['prob_map'].min():.3f}  max={result['prob_map'].max():.3f}")
print(f"   Binary mask: {result['binary_mask'].shape}")


## 🎁 Section 13 — Final Evaluation & Model Save

In [ ]:
# ── Full validation evaluation + confusion matrix ────────────────────────────
@torch.no_grad()
def evaluate_model_full(model, loader, threshold=0.5):
    model.eval()
    all_iou, all_dice = [], []
    all_preds, all_gts = [], []
    for imgs, masks in tqdm(loader, desc='Evaluating'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        preds  = (torch.sigmoid(logits) > threshold).cpu().numpy().astype(np.uint8)
        gts    = masks.cpu().numpy().astype(np.uint8)
        for p, g in zip(preds, gts):
            all_iou.append(compute_iou(p.squeeze(), g.squeeze()))
            all_dice.append(compute_dice(p.squeeze(), g.squeeze()))
            all_preds.extend(p.flatten().tolist())
            all_gts.extend(g.flatten().tolist())
    return float(np.mean(all_iou)), float(np.mean(all_dice)), np.array(all_preds), np.array(all_gts)


val_iou_final, val_dice_final, all_preds, all_gts = evaluate_model_full(seg_model, val_loader)

print('╔══════════════════════════════════════════════╗')
print('║    FINAL EVALUATION — VALIDATION SET         ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Mean IoU  (Jaccard):  {val_iou_final:.4f}               ║')
print(f'║  Mean Dice (F1):       {val_dice_final:.4f}               ║')
print('╚══════════════════════════════════════════════╝')

# ── Confusion matrix visual ───────────────────────────────────────────────────
cm = confusion_matrix(all_gts, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')

ax0 = axes[0]; ax0.set_facecolor('#161b22')
im0 = ax0.imshow(cm_norm, cmap='Blues', vmin=0, vmax=100)
ax0.set_xticks([0,1]); ax0.set_yticks([0,1])
ax0.set_xticklabels(['Pred: Forest', 'Pred: Deforested'], color='white')
ax0.set_yticklabels(['GT: Forest', 'GT: Deforested'], color='white')
for i in range(2):
    for j in range(2):
        ax0.text(j, i, f'{cm[i,j]:,}\n({cm_norm[i,j]:.1f}%)',
                 ha='center', va='center', color='white', fontsize=13, fontweight='bold')
ax0.set_title('🧮 Confusion Matrix (pixel-level)', color='white', fontsize=13, fontweight='bold')
plt.colorbar(im0, ax=ax0, fraction=0.046)

# ── Metrics bar ───────────────────────────────────────────────────────────────
ax1 = axes[1]; ax1.set_facecolor('#161b22')
metrics_names  = ['IoU (Jaccard)', 'Dice / F1']
metrics_values = [val_iou_final, val_dice_final]
bars = ax1.bar(metrics_names, metrics_values, color=['#58a6ff', '#3fb950'], edgecolor='#30363d', width=0.5, linewidth=1.5)
for bar, val in zip(bars, metrics_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', color='white', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1.15); ax1.set_title('📊 Final Validation Metrics', color='white', fontsize=13, fontweight='bold')
ax1.set_ylabel('Score', color='#8b949e', fontsize=11)
ax1.tick_params(colors='#8b949e'); ax1.spines[:].set_color('#30363d')
ax1.grid(alpha=0.2, color='#30363d', axis='y')

plt.suptitle('🎯 Final Model Evaluation', fontsize=16, fontweight='bold', color='white', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Save model artefacts ──────────────────────────────────────────────────────
torch.save({
    'seg_model_state'      : seg_model.state_dict(),
    'temporal_model_state' : temporal_model.state_dict(),
    'in_channels'          : IN_CHANNELS,
    'img_size'             : IMG_SIZE,
    'val_iou'              : val_iou_final,
    'val_dice'             : val_dice_final,
    'history'              : history,
    'has_nir'              : HAS_NIR,
    'use_synthetic'        : USE_SYNTHETIC,
}, 'deforestation_model.pth')

print('💾 Saved: deforestation_model.pth')
print()
print('🎉 Pipeline complete!')
print()
print('Summary:')
print(f'  ✅ Dataset:          {"Synthetic (" + str(N_SYNTHETIC) + " samples)" if USE_SYNTHETIC else str(len(records)) + " real pairs"}')
print(f'  ✅ NDVI:             {"Real NIR" if HAS_NIR else "Green-proxy (RGB only)"}')
print(  '  ✅ U-Net (ResNet34) segmentation model trained')
print(  '  ✅ Change detection via NDVI differencing')
print(  '  ✅ CNN + LSTM temporal model trained')
print(  '  ✅ Future deforestation risk forecasted')
print(  '  ✅ Model compressed (FP16 + L1 pruning)')
print(  '  ✅ Full visualisation dashboard displayed')
print(  '  ✅ Confusion matrix computed')
print(  '  ✅ predict_deforestation() inference function ready')
print()
print(f'Final Metrics → IoU: {val_iou_final:.4f}  |  Dice/F1: {val_dice_final:.4f}')


## 🚀 Section 14 — Streamlit Deployment App

In [ ]:
STREAMLIT_CODE = '''
import streamlit as st
import torch, cv2, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import segmentation_models_pytorch as smp

st.set_page_config(page_title="🌳 Deforestation Detector", layout="wide")
st.title("🌳 AI-Based Deforestation Detection System")
st.markdown("Upload a satellite image (RGB, GeoTIFF, or PNG) to detect deforestation areas.")

@st.cache_resource
def load_model(path="deforestation_model.pth"):
    ckpt  = torch.load(path, map_location="cpu")
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                      in_channels=ckpt.get("in_channels", 5), classes=1, activation=None)
    model.load_state_dict(ckpt["seg_model_state"])
    model.eval()
    return model, ckpt.get("in_channels", 5), ckpt.get("val_iou", 0), ckpt.get("val_dice", 0)

try:
    model, IN_CH, val_iou, val_dice = load_model()
    st.sidebar.success(f"✅ Model loaded  |  IoU: {val_iou:.4f}  |  Dice: {val_dice:.4f}")
except Exception as e:
    st.sidebar.warning(f"Model not found: {e}. Using random weights.")
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=5, classes=1)
    model.eval(); IN_CH = 5

threshold = st.sidebar.slider("Detection Threshold", 0.1, 0.9, 0.5, 0.05)
img_size  = 256
st.sidebar.markdown(f"**Model:** U-Net + ResNet34\n**Input channels:** {IN_CH}")

uploaded = st.file_uploader("Upload Satellite Image", type=["png","jpg","jpeg","tif","tiff"])

if uploaded:
    img    = Image.open(uploaded).convert("RGB")
    img_np = np.array(img).astype(np.float32) / 255.0
    img_np = cv2.resize(img_np, (img_size, img_size))
    ndvi   = ((img_np[:,:,1] - img_np[:,:,0]) / (img_np[:,:,1] + img_np[:,:,0] + 1e-8))[:,:,None]
    sar    = np.random.uniform(0.1, 0.6, (img_size, img_size, 1)).astype(np.float32)
    stack  = np.concatenate([img_np, ndvi, sar], axis=-1)[:,:,:IN_CH]
    tensor = torch.from_numpy(stack.transpose(2,0,1)).float().unsqueeze(0)
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).squeeze().numpy()
    binary = (prob > threshold).astype(np.uint8)
    defor_pct = binary.mean() * 100

    col1, col2, col3, col4 = st.columns(4)
    with col1: st.image(img, caption="Original", use_container_width=True)
    with col2:
        fig, ax = plt.subplots(); ax.imshow(ndvi[:,:,0], cmap="RdYlGn", vmin=-1, vmax=1); ax.axis("off")
        st.pyplot(fig); st.caption("NDVI Map")
    with col3:
        prob_cmap = LinearSegmentedColormap.from_list("prob", ["#0d1117","#ffa657","#e74c3c"])
        fig2, ax2 = plt.subplots(); ax2.imshow(prob, cmap=prob_cmap, vmin=0, vmax=1); ax2.axis("off")
        st.pyplot(fig2); st.caption("Probability Map")
    with col4:
        fig3, ax3 = plt.subplots(); ax3.imshow(binary, cmap="hot"); ax3.axis("off")
        st.pyplot(fig3); st.caption(f"Deforested: {defor_pct:.1f}%")

    if defor_pct < 20:
        st.success(f"🟢 LOW risk — {defor_pct:.1f}% deforested")
    elif defor_pct < 50:
        st.warning(f"🟡 MODERATE risk — {defor_pct:.1f}% deforested")
    else:
        st.error(f"🔴 HIGH risk — {defor_pct:.1f}% deforested")
'''

with open('streamlit_app.py', 'w') as f:
    f.write(STREAMLIT_CODE.strip())
print('✅ Streamlit app saved → streamlit_app.py')
print('   Run with: streamlit run streamlit_app.py')
